# MADP for Sri Lankan conveyancing documents

An implementation of **MADP: A Multi-Agent Pipeline for Sustainable Document
Processing with Human-in-the-Loop** (Gosmar & Zenezini, arXiv:2605.17159),
applied to the real matter bundles in `ocr-benchmark/cases/`.

Everything is in this one notebook: the five agents, the PFTFI feedback loop,
the human-in-the-loop queue, the metrics, and the ablation. It imports only the
already-tested low-level helpers (`render`, `normalize`, `contract`) so page
rendering and score normalization stay identical to the benchmark.

## The pipeline

| # | Agent | Paper's role | What it does here |
|---|-------|--------------|-------------------|
| 1 | Classificator | document type + routing | RTA registry kind per document |
| 2 | Splitter | separate logical units in a batch file | find instrument boundaries inside a bundle |
| 3 | Parser | layout → structured markdown | OCR + reading order → markdown |
| 4 | Extraction | LLM → JSON fields + confidence | registry fields, consensus across backends |
| 5 | Validator + PFTFI | atomic checks, escalate, learn | RTA checks, HITL queue, prompt/rule feedback |

## Three honest adaptations

The paper processes **invoices** with a text layer. This corpus is different in
ways that change the design, so the differences are stated rather than hidden.

**1. Every PDF here is image-only.** All 19 PDFs carry no extractable text
(≤ 84 non-whitespace characters across the first four pages). The paper's Parser
Agent uses Docling's layout analysis, which is its single largest reported gain
(+17.5 pp). Docling with OCR disabled returns nothing on these scans, so the
Parser Agent here is **OCR-backed**: recognise text and boxes, then rebuild
reading order and emit markdown. That keeps the paper's actual mechanism —
structure-aware markdown instead of a raw text dump — on inputs that have no
digital structure to read.

**2. The atomic consistency checks are domain-swapped.** The paper checks
`subtotal + tax = total`, VAT rates, ISO-4217 currency. None of that exists on a
title certificate. The equivalent conveyancing invariants are used instead: NIC
format, extent expressed in hectares, cadastral identifiers numeric, transferred
extent not exceeding parcel extent, and consideration agreeing between the
instrument and the cheque. This is the paper's idea — deterministic symbolic
checks that elevate confidence and catch hallucinations — applied to this domain.

**3. No LLM backend is reachable.** The configured Gemini key returns 401 and
Vertex is not enabled on the project, so the Extraction Agent runs on a
deterministic rule-based backend. The LLM path is written and wired; it activates
the moment a working key exists. Expect low recall until then, especially on
Sinhala: the available OCR engine recognises Latin script only, so Sinhala pages
produce garbage text. **That is a measurement of the current setup, not a
property of MADP**, and the notebook labels it as such throughout.

In [ ]:
from __future__ import annotations

import json, re, sys, time, unicodedata
from collections import Counter, defaultdict
from dataclasses import dataclass, field, asdict
from pathlib import Path
from typing import Any, Callable, Iterable, Sequence

sys.path.insert(0, str(Path.cwd() if Path.cwd().name == "ocr-benchmark" else Path.cwd() / "ocr-benchmark"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import config, contract, engines, normalize, render, viz

viz.use_style()
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 160)

REGISTRY = json.loads((config.SCHEMAS / "registry-fields.json").read_text(encoding="utf-8"))["kinds"]
CRITICAL = set(json.loads((config.SCHEMAS / "critical-fields.json").read_text(encoding="utf-8"))["critical"])
FIELD_LABELS = {f["key"]: f["label"] for fs in REGISTRY.values() for f in fs}

print("corpus:")
for case in config.case_dirs():
    print(f"  {case.name:45s} {len(config.case_documents(case)):3d} documents")
print()
for engine, state in engines.capabilities().items():
    print(f"  {engine:10s} {state}")

## Pipeline configuration

The paper sets escalation thresholds "typically between 80–90% based on document
type, field criticality, and organizational risk tolerance". Criticality is
already declared for this domain, so critical fields get the strictest threshold:
a wrong parcel number is not a cost problem, it is a wrong deed.

In [ ]:
@dataclass
class MADPConfig:
    dpi: int = 200
    render_variant: str = "full"          # Parser Agent benefits from preprocessing
    classify_threshold: float = 0.70      # below this, the document is escalated
    field_threshold: float = 0.85         # paper's 80-90% band
    critical_threshold: float = 0.95      # stricter: a wrong identifier is a wrong deed
    atomic_pass_confidence: float = 0.99  # paper elevates to near-certainty on check pass
    consensus_bonus: float = 0.10         # agreement across backends raises confidence
    max_pages_per_doc: int | None = 6     # keep the demo run tractable; None = all
    review_seconds_flagged: int = 45      # paper's measured HITL review time
    review_seconds_manual: int = 120      # paper's fully-manual baseline

    # Which agents are active. The ablation flips these one at a time.
    use_classificator: bool = True
    use_splitter: bool = True
    use_parser: bool = True
    use_validator: bool = True
    use_hitl: bool = True

CFG = MADPConfig()
print(CFG)

## Agent 1 — Classificator

Determines the document kind and its routing decision. Below
`classify_threshold` the document is escalated rather than guessed at, which is
the paper's "degrade gracefully to manual inspection rather than producing
unreliable extractions".

Two backends: the LLM classifier when a key works, otherwise a transparent
keyword-evidence scorer over the parsed text. The offline scorer is deliberately
weak-but-honest — it reports low confidence when it sees little evidence, which
sends the document to a human instead of inventing a kind.

In [ ]:
KIND_EVIDENCE: dict[str, list[str]] = {
    "title-certificate": ["certificate of title", "title certificate", "cadastral", "parcel",
                          "hectare", "bim saviya", "registration of title", "class of title"],
    "form8-instrument": ["form 8", "instrument of transfer", "transferor", "transferee",
                         "consideration", "notary", "attestation", "section 43"],
    "survey-plan": ["survey plan", "licensed surveyor", "lot no", "surveyor", "scale",
                    "bearing", "boundaries", "plan no"],
    "identity-card": ["national identity card", "identity card", "date of birth", "nic"],
    "other": [],
}

def classify_offline(text: str) -> tuple[str, float, dict[str, int]]:
    """Keyword-evidence scorer. Transparent by design: it returns its evidence."""
    low = normalize.platform_norm(text or "")
    hits = {kind: sum(1 for kw in kws if kw in low) for kind, kws in KIND_EVIDENCE.items() if kws}
    if not any(hits.values()):
        return "other", 0.0, hits
    best = max(hits, key=lambda k: hits[k])
    total = sum(hits.values())
    # Confidence = share of evidence for the winner, damped by how little evidence
    # there is overall. Two keyword hits should not look like certainty.
    share = hits[best] / total
    evidence_weight = min(1.0, hits[best] / 4.0)
    return best, round(share * evidence_weight, 3), hits

def classificator_agent(text: str, page_image=None, cfg: MADPConfig = CFG) -> dict[str, Any]:
    ok, _why = engines.gemini_available()
    if ok and page_image is not None:
        try:
            page = render.RenderedPage(1, page_image, cfg.dpi, "original", contract.IDENTITY, False)
            kind, conf, reading = engines.gemini_classify_page(page)
            if not reading.errors:
                return {"kind": kind, "confidence": conf, "source": "gemini", "evidence": {}}
        except Exception as exc:
            pass  # fall through to the offline scorer
    kind, conf, hits = classify_offline(text)
    return {"kind": kind, "confidence": conf, "source": "keyword-evidence", "evidence": hits}

# Sanity check on invented text, so no client content appears in the notebook.
demo = "CERTIFICATE OF TITLE cadastral map 520005 parcel 0021 extent 0.0153 hectare class of title"
print(classificator_agent(demo))
print(classificator_agent("illegible smudge"))

## Agent 2 — Splitter

The paper's splitter separates "individual invoices within a batch file" using
visual features and semantic analysis, because context mixing across document
boundaries causes extraction errors.

That problem is acute here. One bundle is 16 pages, another 29, and one is a
168-page court file. Production currently classifies a whole document from page 1
only, so everything after the first instrument is read under the wrong schema —
exactly the failure the splitter exists to prevent.

Boundaries are detected from three signals: a page whose kind evidence disagrees
with the running segment, a page that looks like a cover or form header, and a
large drop in ink coverage (blank or near-blank separator pages).

In [ ]:
HEADER_MARKERS = ["form 8", "certificate of title", "survey plan", "instrument of transfer",
                  "affidavit", "petition", "in the supreme court", "receipt", "resolution"]

def splitter_agent(pages: list[dict[str, Any]], cfg: MADPConfig = CFG) -> list[dict[str, Any]]:
    """Group pages into logical documents. Returns one segment per logical unit."""
    if not cfg.use_splitter:
        return [{"segment": 0, "pages": [p["page_no"] for p in pages],
                 "boundary_reasons": ["splitter-disabled"]}]

    segments: list[dict[str, Any]] = []
    current: list[int] = []
    reasons: list[str] = []
    running_kind: str | None = None

    for page in pages:
        text = normalize.platform_norm(page.get("text", ""))
        kind, conf, _ = classify_offline(page.get("text", ""))
        is_header = any(m in text for m in HEADER_MARKERS)
        kind_shift = running_kind is not None and kind != "other" and conf >= 0.4 and kind != running_kind
        blank = page.get("blank", False)

        boundary = current and (blank or (is_header and len(current) > 1) or kind_shift)
        if boundary:
            why = "blank-separator" if blank else ("header-marker" if is_header else "kind-change")
            segments.append({"segment": len(segments), "pages": current, "boundary_reasons": reasons or [why]})
            current, reasons = [], [why]
            running_kind = None
        if not blank:
            current.append(page["page_no"])
            if kind != "other" and conf >= 0.4:
                running_kind = kind
    if current:
        segments.append({"segment": len(segments), "pages": current, "boundary_reasons": reasons or ["end"]})
    return segments or [{"segment": 0, "pages": [p["page_no"] for p in pages], "boundary_reasons": ["single"]}]

fake = [
    {"page_no": 1, "text": "CERTIFICATE OF TITLE cadastral parcel hectare", "blank": False},
    {"page_no": 2, "text": "schedule of the parcel continued", "blank": False},
    {"page_no": 3, "text": "FORM 8 INSTRUMENT OF TRANSFER transferor transferee notary", "blank": False},
    {"page_no": 4, "text": "consideration paid attestation", "blank": False},
]
for seg in splitter_agent(fake):
    print(seg)

## Agent 3 — Parser

The paper's largest single gain (+17.5 pp). It converts raw pages into structured
markdown: identify text regions, determine reading order, recognise tables, and
emit a hierarchical representation that preserves spatial relationships while
cutting token count ~35% against raw OCR.

Since every PDF here is image-only, the parser is OCR-backed. Reading order is
reconstructed by grouping recognised boxes into lines (by vertical overlap) and
then into columns, which is the part that actually matters: raw OCR output is
ordered by detection confidence, not by how a human reads the page.

In [ ]:
_OCR_ENGINE = None

def _ocr():
    global _OCR_ENGINE
    if _OCR_ENGINE is None:
        from rapidocr import RapidOCR
        _OCR_ENGINE = RapidOCR()
    return _OCR_ENGINE

def _ocr_cache_path(doc_id: str, page_no: int, dpi: int, variant: str) -> Path:
    safe = "".join(c if c.isalnum() or c in "-_." else "-" for c in doc_id)
    return config.RENDERS / "ocr" / safe / f"p{page_no:04d}@{dpi}-{variant}.json"

def ocr_page(page: render.RenderedPage, doc_id: str = "") -> list[dict[str, Any]]:
    """Recognised boxes with normalized geometry, cached to disk.

    The ablation runs six configurations over the same pages, so without a cache
    every page is recognised six times. OCR dominates the runtime, so this is the
    difference between a ~25 minute run and a ~5 minute one.
    """
    cache = _ocr_cache_path(doc_id or "adhoc", page.page_no, page.dpi, page.variant)
    if cache.is_file():
        try:
            return json.loads(cache.read_text(encoding="utf-8"))
        except json.JSONDecodeError:
            pass

    result = _ocr()(np.asarray(page.image.convert("RGB")))
    boxes = getattr(result, "boxes", None)
    txts = getattr(result, "txts", None) or []
    scores = getattr(result, "scores", None) or []
    w, h = page.size
    out = []
    if boxes is not None:
        for i, poly in enumerate(boxes):
            out.append({
                "bbox": list(contract.bbox_from_polygon(poly, w, h)),
                "text": str(txts[i]) if i < len(txts) else "",
                "confidence": float(scores[i]) if i < len(scores) else 0.0,
            })
    cache.parent.mkdir(parents=True, exist_ok=True)
    cache.write_text(json.dumps(out, ensure_ascii=False), encoding="utf-8")
    return out

def reading_order_markdown(blocks: list[dict[str, Any]], line_tol: float = 0.012) -> str:
    """Group boxes into visual lines, order them, and emit markdown.

    This is the Parser Agent's real work. Raw OCR returns boxes in detection
    order; a human reads top-to-bottom, left-to-right. Without this the LLM sees
    a shuffled bag of phrases and silently mis-associates labels with values.
    """
    if not blocks:
        return ""
    items = sorted(blocks, key=lambda b: (b["bbox"][1], b["bbox"][0]))
    lines: list[list[dict[str, Any]]] = []
    for block in items:
        y_centre = (block["bbox"][1] + block["bbox"][3]) / 2
        placed = False
        for line in lines:
            ref = (line[0]["bbox"][1] + line[0]["bbox"][3]) / 2
            if abs(y_centre - ref) <= line_tol:
                line.append(block)
                placed = True
                break
        if not placed:
            lines.append([block])

    rendered: list[str] = []
    for line in lines:
        line.sort(key=lambda b: b["bbox"][0])
        texts = [b["text"].strip() for b in line if b["text"].strip()]
        if not texts:
            continue
        # Several boxes far apart on one line is a table row, not a sentence.
        gaps = [line[i + 1]["bbox"][0] - line[i]["bbox"][2] for i in range(len(line) - 1)]
        if len(texts) > 2 and gaps and max(gaps) > 0.05:
            rendered.append("| " + " | ".join(texts) + " |")
        else:
            rendered.append(" ".join(texts))
    return "\n".join(rendered)

def parser_agent(page: render.RenderedPage, cfg: MADPConfig = CFG,
                 doc_id: str = "") -> dict[str, Any]:
    began = time.time()
    blocks = ocr_page(page, doc_id)
    raw = "\n".join(b["text"] for b in blocks)
    markdown = reading_order_markdown(blocks) if cfg.use_parser else raw
    conf = float(np.mean([b["confidence"] for b in blocks])) if blocks else 0.0
    return {
        "page_no": page.page_no,
        "text": markdown,
        "raw_text": raw,
        "blocks": blocks,
        "ocr_confidence": round(conf, 4),
        "blank": page.blank,
        "script": normalize.script_of(raw),
        "seconds": round(time.time() - began, 2),
    }

print("parser ready — reading-order demo on synthetic boxes:")
demo_blocks = [
    {"bbox": (0.55, 0.10, 0.75, 0.12), "text": "0021", "confidence": 0.9},
    {"bbox": (0.10, 0.10, 0.40, 0.12), "text": "Parcel No", "confidence": 0.9},
    {"bbox": (0.10, 0.20, 0.40, 0.22), "text": "Extent", "confidence": 0.9},
    {"bbox": (0.55, 0.20, 0.75, 0.22), "text": "0.0153 ha", "confidence": 0.9},
]
print(reading_order_markdown(demo_blocks))

## Agent 4 — Extraction

Schema-constrained field extraction with a confidence per field, plus the paper's
**parallel extraction strategy**: run more than one backend over the same input,
raise confidence where they agree, and flag disagreements for review.

Two backends are implemented. The LLM backend is the paper's design and is wired
but inert without credentials. The rule backend is deterministic pattern matching
over the parsed markdown — it is what makes the whole pipeline runnable and
measurable offline, and it doubles as the second voter for consensus.

In [ ]:
# Deterministic extractors. Each returns (value, confidence).
# Confidence reflects how specific the evidence was, not how much we like it.
def _search(patterns: Sequence[str], text: str, conf: float = 0.75):
    for pattern in patterns:
        m = re.search(pattern, text, re.I)
        if m:
            value = (m.group(1) if m.groups() else m.group(0)).strip(" .:;|")
            if value:
                return value, conf
    return None, 0.0

RULES: dict[str, Callable[[str], tuple[str | None, float]]] = {
    "titleCertificateNo": lambda t: _search([r"(?:title\s*certificate|certificate)\s*(?:no|number|#)?\s*[:.\-]?\s*([0-9]{6,14})"], t, 0.8),
    "cadastralMapNo":     lambda t: _search([r"cadastral\s*(?:map)?\s*(?:no|number)?\s*[:.\-]?\s*([0-9]{4,10})"], t, 0.8),
    "blockNo":            lambda t: _search([r"block\s*(?:no|number)?\s*[:.\-]?\s*([0-9]{1,3})"], t, 0.7),
    "sheetNo":            lambda t: _search([r"sheet\s*(?:no|number)?\s*[:.\-]?\s*([0-9]{1,3})"], t, 0.7),
    "parcelNo":           lambda t: _search([r"parcel\s*(?:no|number)?\s*[:.\-]?\s*([0-9]{1,6})"], t, 0.75),
    "extent":             lambda t: _search([r"([0-9]+\.[0-9]+\s*(?:ha|hectares?))"], t, 0.75),
    "surveyPlanNo":       lambda t: _search([r"plan\s*(?:no|number)?\s*[:.\-]?\s*([0-9]{3,6}[A-Za-z]?)"], t, 0.7),
    "lotNo":              lambda t: _search([r"lot\s*(?:no|number)?\s*[:.\-]?\s*([0-9]{1,4}[A-Za-z]?)"], t, 0.7),
    "consideration":      lambda t: _search([r"(rs\.?\s*[0-9][0-9,]{4,}(?:/[=\-])?)", r"([0-9]{1,3}(?:,[0-9]{3}){2,})"], t, 0.7),
    "transferorNic":      lambda t: _search([r"\b([0-9]{9}[VXvx]|[0-9]{12})\b"], t, 0.5),
    "transfereeNic":      lambda t: _search([r"\b([0-9]{9}[VXvx]|[0-9]{12})\b"], t, 0.5),
    "holderDateOfBirth":  lambda t: _search([r"\b([0-9]{4}-[0-9]{2}-[0-9]{2})\b"], t, 0.6),
    "district":           lambda t: _search([r"district\s*[:.\-]?\s*([A-Za-z]{3,20})"], t, 0.6),
    "notaryCode":         lambda t: _search([r"notary\s*(?:code|no)?\s*[:.\-]?\s*([0-9]{3,8})"], t, 0.6),
    "surveyorRegistration": lambda t: _search([r"(?:reg(?:istration)?|licensed\s*surveyor)\s*(?:no)?\s*[:.\-]?\s*([0-9]{3,8})"], t, 0.6),
}

def extract_rules(text: str, keys: Sequence[str]) -> dict[str, dict[str, Any]]:
    out = {}
    for key in keys:
        rule = RULES.get(key)
        if rule is None:
            continue
        value, conf = rule(text)
        if value is not None:
            out[key] = {"value": value, "confidence": conf, "backend": "rules"}
    return out

def extract_llm(page: render.RenderedPage, keys: Sequence[str]) -> dict[str, dict[str, Any]]:
    ok, _ = engines.gemini_available()
    if not ok:
        return {}
    try:
        reading = engines.gemini_read_page(page, keys)
    except Exception:
        return {}
    if reading.errors and not reading.fields:
        return {}
    return {
        f.key: {"value": f.value, "confidence": f.model_confidence or 0.8, "backend": "llm"}
        for f in reading.fields if f.value is not None
    }

def extraction_agent(text: str, keys: Sequence[str], page=None, cfg: MADPConfig = CFG) -> dict[str, dict[str, Any]]:
    """Parallel extraction with consensus voting (paper section 3.5)."""
    candidates = [extract_rules(text, keys)]
    if page is not None:
        llm = extract_llm(page, keys)
        if llm:
            candidates.append(llm)

    merged: dict[str, dict[str, Any]] = {}
    for key in keys:
        votes = [c[key] for c in candidates if key in c]
        if not votes:
            continue
        if len(votes) == 1:
            merged[key] = dict(votes[0]) | {"agreement": [votes[0]["backend"]]}
            continue
        norms = {normalize.platform_norm(str(v["value"])) for v in votes}
        best = max(votes, key=lambda v: v["confidence"])
        if len(norms) == 1:
            merged[key] = dict(best) | {
                "confidence": min(1.0, best["confidence"] + cfg.consensus_bonus),
                "agreement": [v["backend"] for v in votes],
            }
        else:
            # Disagreement is a review trigger, not a tie-break to hide.
            merged[key] = dict(best) | {
                "confidence": min(best["confidence"], 0.5),
                "agreement": [],
                "disagreement": sorted(norms),
            }
    return merged

print(extract_rules("Parcel No 0021 Extent 0.0153 ha Cadastral Map No 520005 Rs.4,819,500/=",
                    ["parcelNo", "extent", "cadastralMapNo", "consideration"]))

## Agent 5 — Validator, atomic checks, and escalation

The paper's validator enforces format rules, numeric relationships, plausible
ranges and cross-field dependencies, then **elevates confidence to near-certainty
when the deterministic checks pass** — combining symbolic reasoning with neural
extraction so that moderate LLM confidence does not force a human review.

The invoice arithmetic is replaced with conveyancing invariants. The important
one is the last: an instrument cannot transfer more land than the parcel contains.

In [ ]:
def _hectares(value: str | None) -> float | None:
    if not value:
        return None
    m = re.search(r"([0-9]+(?:\.[0-9]+)?)", value)
    return float(m.group(1)) if m else None

def _money(value: str | None) -> float | None:
    if not value:
        return None
    digits = re.sub(r"[^0-9.]", "", value.replace(",", ""))
    try:
        return float(digits) if digits else None
    except ValueError:
        return None

FORMAT_CHECKS: dict[str, Callable[[str], bool]] = {
    "transferorNic": normalize.is_valid_nic,
    "transfereeNic": normalize.is_valid_nic,
    "holderDateOfBirth": normalize.is_valid_iso_date,
    "extent": normalize.is_valid_extent,
    "extentSubjectToTransfer": normalize.is_valid_extent,
    "cadastralMapNo": normalize.is_digits,
}

def atomic_checks(fields: dict[str, dict[str, Any]]) -> list[dict[str, Any]]:
    """Deterministic conveyancing invariants. Each returns pass/fail plus why."""
    results: list[dict[str, Any]] = []

    for key, check in FORMAT_CHECKS.items():
        if key in fields and fields[key].get("value"):
            ok = check(str(fields[key]["value"]))
            results.append({"check": f"format:{key}", "passed": ok, "fields": [key],
                            "detail": "" if ok else "does not match the expected format"})

    parcel = _hectares(fields.get("extent", {}).get("value"))
    transferred = _hectares(fields.get("extentSubjectToTransfer", {}).get("value"))
    if parcel is not None and transferred is not None:
        ok = transferred <= parcel + 1e-9
        results.append({"check": "extent:transferred<=parcel", "passed": ok,
                        "fields": ["extent", "extentSubjectToTransfer"],
                        "detail": "" if ok else "transfers more land than the parcel contains"})

    consideration = _money(fields.get("consideration", {}).get("value"))
    if consideration is not None:
        ok = 1_000 <= consideration <= 10_000_000_000
        results.append({"check": "range:consideration", "passed": ok, "fields": ["consideration"],
                        "detail": "" if ok else "outside a plausible range"})

    a, b = fields.get("transferorNic", {}).get("value"), fields.get("transfereeNic", {}).get("value")
    if a and b:
        ok = normalize.platform_norm(str(a)) != normalize.platform_norm(str(b))
        results.append({"check": "distinct:parties", "passed": ok,
                        "fields": ["transferorNic", "transfereeNic"],
                        "detail": "" if ok else "transferor and transferee share one NIC"})
    return results

def validator_agent(fields: dict[str, dict[str, Any]], parsed_text: str,
                    cfg: MADPConfig = CFG,
                    required: Sequence[str] = ()) -> dict[str, Any]:
    checks = atomic_checks(fields) if cfg.use_validator else []
    passed = {c["check"]: c["passed"] for c in checks}
    failed_fields = {f for c in checks if not c["passed"] for f in c["fields"]}

    escalations: list[dict[str, Any]] = []

    # Silence is not success. A document the pipeline could not read at all must
    # go to a human, otherwise it counts as straight-through processing purely
    # because there was nothing to disagree with — which would make an empty
    # extractor look like the most automated configuration.
    required_critical = [k for k in required if k in CRITICAL]
    missing_critical = [k for k in required_critical
                        if not (fields.get(k) or {}).get("value")]
    if required and not any((entry or {}).get("value") for entry in fields.values()):
        escalations.append({"key": "__document__", "reasons": ["no-fields-extracted"],
                            "confidence": 0.0})
    elif missing_critical:
        escalations.append({"key": "__document__",
                            "reasons": [f"missing-critical:{','.join(missing_critical[:4])}"],
                            "confidence": 0.0})

    for key, entry in fields.items():
        value = entry.get("value")
        # Grounding: a value absent from the page text is a hallucination candidate.
        entry["grounded"] = normalize.value_in_text(str(value), parsed_text) if value else False

        if cfg.use_validator and key not in failed_fields:
            relevant = [c for c in checks if key in c["fields"]]
            if relevant and all(c["passed"] for c in relevant):
                # Paper: atomic pass elevates confidence to near-certainty.
                entry["confidence"] = max(entry.get("confidence", 0.0), cfg.atomic_pass_confidence)
                entry["elevated"] = True

        threshold = cfg.critical_threshold if key in CRITICAL else cfg.field_threshold
        entry["threshold"] = threshold
        reasons = []
        if key in failed_fields:
            reasons.append("failed-atomic-check")
        if entry.get("confidence", 0.0) < threshold:
            reasons.append("below-confidence-threshold")
        if entry.get("disagreement"):
            reasons.append("backend-disagreement")
        if value and not entry["grounded"]:
            reasons.append("not-verbatim-in-page")
        if reasons:
            escalations.append({"key": key, "reasons": reasons,
                                "confidence": entry.get("confidence", 0.0)})

    return {"checks": checks, "checks_passed": passed, "escalations": escalations,
            "requires_human": bool(escalations)}

demo_fields = {
    "extent": {"value": "0.0153 ha", "confidence": 0.7},
    "extentSubjectToTransfer": {"value": "0.0159 ha", "confidence": 0.7},
    "transfereeNic": {"value": "199012345678", "confidence": 0.6},
}
demo_text = "extent 0.0153 ha transferred 0.0159 ha nic 199012345678"
for check in validator_agent(dict(demo_fields), demo_text)["checks"]:
    print(f"{'PASS' if check['passed'] else 'FAIL'}  {check['check']:34s} {check['detail']}")

# An empty extraction must escalate, not sail through.
empty = validator_agent({}, "", required=["parcelNo", "extent"])
print("\nempty extraction ->", empty["escalations"][0]["reasons"],
      "| requires_human =", empty["requires_human"])

## PFTFI — Prompt Fine Tuning with Feedback Inheritance

The paper's contribution. Human corrections are captured as structured feedback,
converted into prompt examples and parser hints, versioned, and — the
*inheritance* part — applied to documents still pending in the queue that share
the same characteristics, not just to future ones.

No model weights are touched. Feedback splits two ways, exactly as described:
semantic corrections refine the Extraction prompt, layout corrections refine the
Parser configuration.

In [ ]:
@dataclass
class Correction:
    doc_id: str
    kind: str
    key: str
    original: str | None
    corrected: str
    cause: str  # "semantic" (extraction) or "layout" (parser)

class PFTFI:
    """Versioned prompt/rule store driven by human corrections."""

    def __init__(self) -> None:
        self.version = 0
        self.corrections: list[Correction] = []
        self.extraction_examples: dict[str, list[str]] = defaultdict(list)
        self.parser_hints: list[str] = []
        self.learned_patterns: dict[str, list[str]] = defaultdict(list)
        self.history: list[dict[str, Any]] = []

    def capture(self, corrections: list[Correction]) -> None:
        if not corrections:
            return
        self.corrections.extend(corrections)
        for c in corrections:
            if c.cause == "layout":
                hint = f"On {c.kind}, {c.key} is often mis-read; check reading order around it."
                if hint not in self.parser_hints:
                    self.parser_hints.append(hint)
                continue
            example = f'{c.kind}.{c.key} -> "{c.corrected}"'
            if example not in self.extraction_examples[c.kind]:
                self.extraction_examples[c.kind].append(example)
            # Generalise the correction into a reusable pattern so the fix
            # transfers to other documents rather than memorising one answer.
            pattern = self._generalise(c.corrected)
            if pattern and pattern not in self.learned_patterns[c.key]:
                self.learned_patterns[c.key].append(pattern)
        self.version += 1
        self.history.append({
            "version": self.version,
            "corrections": len(corrections),
            "cumulative": len(self.corrections),
            "patterns": sum(len(v) for v in self.learned_patterns.values()),
        })

    @staticmethod
    def _generalise(value: str) -> str | None:
        """Turn a corrected value into a shape-matching regex."""
        v = value.strip()
        if not v:
            return None
        if re.fullmatch(r"[0-9]+", v):
            return rf"\b([0-9]{{{len(v)}}})\b"
        if re.fullmatch(r"[0-9]+\.[0-9]+\s*(ha|hectares?)", v, re.I):
            return r"([0-9]+\.[0-9]+\s*(?:ha|hectares?))"
        if re.fullmatch(r"[0-9]{9}[VXvx]|[0-9]{12}", v):
            return r"\b([0-9]{9}[VXvx]|[0-9]{12})\b"
        return None

    def prompt_suffix(self, kind: str) -> str:
        examples = self.extraction_examples.get(kind, [])
        if not examples:
            return ""
        return ("\nPreviously corrected by a reviewer on this document type:\n  "
                + "\n  ".join(examples[-8:]))

    def apply_inheritance(self, text: str, keys: Sequence[str],
                          existing: dict[str, dict[str, Any]]) -> dict[str, dict[str, Any]]:
        """Feedback inheritance: retry still-missing fields with learned patterns."""
        out = dict(existing)
        for key in keys:
            if key in out and out[key].get("value"):
                continue
            for pattern in self.learned_patterns.get(key, []):
                m = re.search(pattern, text, re.I)
                if m:
                    out[key] = {"value": m.group(1), "confidence": 0.7,
                                "backend": "pftfi", "agreement": ["pftfi"]}
                    break
        return out

PFTFI_STORE = PFTFI()
PFTFI_STORE.capture([Correction("demo", "title-certificate", "parcelNo", "021", "0021", "semantic")])
print("version:", PFTFI_STORE.version)
print("learned:", dict(PFTFI_STORE.learned_patterns))
print(PFTFI_STORE.prompt_suffix("title-certificate"))
PFTFI_STORE = PFTFI()  # reset before the real run

## Orchestrator

Runs the five agents in sequence over one document and records everything needed
for the metrics: which agent escalated, how long each stage took, and whether the
document achieved straight-through processing.

In [ ]:
def field_keys_for(kind: str) -> list[str]:
    return [f["key"] for f in REGISTRY.get(kind, [])]

def process_document(path: Path, cfg: MADPConfig = CFG,
                     pftfi: PFTFI | None = None) -> dict[str, Any]:
    began = time.time()
    doc_id = f"{path.parent.name}/{path.name}"
    timings: dict[str, float] = {}
    escalated_by: list[str] = []

    # --- render -------------------------------------------------------------
    t = time.time()
    variant = cfg.render_variant if cfg.use_parser else "original"
    pages = render.render_doc(path, dpi=cfg.dpi, variant=variant,
                              page_limit=cfg.max_pages_per_doc)
    timings["render"] = round(time.time() - t, 2)

    # --- Agent 3: Parser ----------------------------------------------------
    t = time.time()
    parsed = [parser_agent(p, cfg, doc_id) for p in pages]
    timings["parse"] = round(time.time() - t, 2)
    full_text = "\n".join(p["text"] for p in parsed)

    # --- Agent 2: Splitter --------------------------------------------------
    t = time.time()
    segments = splitter_agent(parsed, cfg)
    timings["split"] = round(time.time() - t, 2)

    # --- Agent 1: Classificator --------------------------------------------
    t = time.time()
    if cfg.use_classificator:
        routing = classificator_agent(full_text, pages[0].image if pages else None, cfg)
    else:
        # Baseline: no classifier, so every field of every kind is requested.
        routing = {"kind": "__all__", "confidence": 0.0, "source": "disabled", "evidence": {}}
    timings["classify"] = round(time.time() - t, 2)
    if cfg.use_classificator and routing["confidence"] < cfg.classify_threshold:
        escalated_by.append("classificator")

    keys = (sorted({f["key"] for fs in REGISTRY.values() for f in fs})
            if routing["kind"] == "__all__" else field_keys_for(routing["kind"]))

    # --- Agent 4: Extraction ------------------------------------------------
    t = time.time()
    fields = extraction_agent(full_text, keys, pages[0] if pages else None, cfg)
    if pftfi is not None:
        fields = pftfi.apply_inheritance(full_text, keys, fields)
    timings["extract"] = round(time.time() - t, 2)

    # --- Agent 5: Validator -------------------------------------------------
    t = time.time()
    validation = validator_agent(fields, full_text, cfg, required=keys)
    timings["validate"] = round(time.time() - t, 2)
    if validation["requires_human"]:
        escalated_by.append("validator")

    return {
        "doc_id": doc_id,
        "matter": path.parent.name,
        "pages": len(pages),
        "routing": routing,
        "segments": segments,
        "fields": fields,
        "validation": validation,
        "escalated_by": escalated_by,
        "straight_through": not escalated_by,
        "ocr_confidence": round(float(np.mean([p["ocr_confidence"] for p in parsed])) if parsed else 0.0, 4),
        "scripts": dict(Counter(p["script"] for p in parsed)),
        "text_chars": len(re.sub(r"\s+", "", full_text)),
        "timings": timings,
        "seconds": round(time.time() - began, 2),
        "_text": full_text,   # kept in memory only; never written to a tracked file
    }

print("orchestrator ready")

## Run the pipeline

Defaults to the labelled matter so results can be scored. Set `RUN_ALL_MATTERS`
to process all 38 documents — at roughly 7 seconds of OCR per page that is a
long run, which is why `max_pages_per_doc` is capped.

In [ ]:
RUN_ALL_MATTERS = False

if RUN_ALL_MATTERS:
    documents = [d for case in config.case_dirs() for d in config.case_documents(case)]
else:
    labelled = [c for c in config.case_dirs() if config.expected_fields_path(c).is_file()]
    documents = [d for case in labelled for d in config.case_documents(case)]

print(f"processing {len(documents)} documents (max {CFG.max_pages_per_doc} pages each)\n")
results: list[dict[str, Any]] = []
for i, doc in enumerate(documents, 1):
    out = process_document(doc, CFG, PFTFI_STORE)
    results.append(out)
    print(f"[{i:2d}/{len(documents)}] {out['doc_id'].split('/')[-1][:44]:46s} "
          f"kind={out['routing']['kind'][:18]:18s} conf={out['routing']['confidence']:.2f} "
          f"fields={len(out['fields']):2d} STP={'Y' if out['straight_through'] else 'n'} "
          f"{out['seconds']:5.1f}s")

print(f"\ntotal {sum(r['seconds'] for r in results):.0f}s, "
      f"{sum(r['pages'] for r in results)} pages")

In [ ]:
summary = pd.DataFrame([
    {
        "document": r["doc_id"].split("/")[-1][:40],
        "pages": r["pages"],
        "segments": len(r["segments"]),
        "kind": r["routing"]["kind"],
        "kind_conf": r["routing"]["confidence"],
        "ocr_conf": r["ocr_confidence"],
        "chars": r["text_chars"],
        "fields": len(r["fields"]),
        "escalations": len(r["validation"]["escalations"]),
        "STP": r["straight_through"],
        "secs": r["seconds"],
    }
    for r in results
])
summary

## Human-in-the-loop queue

The paper's validation GUI shows the document beside the extracted JSON,
highlights low-confidence fields, and captures corrections as structured feedback
(original value, corrected value, field, document type). Here that is a review
table plus a correction-capture function — the data contract matters, the widget
does not.

Reviewers correct the extracted metadata, never the source document.

In [ ]:
queue_rows = []
for r in results:
    for esc in r["validation"]["escalations"]:
        entry = r["fields"].get(esc["key"], {})
        queue_rows.append({
            "doc": r["doc_id"].split("/")[-1][:34],
            "kind": r["routing"]["kind"],
            "field": esc["key"],
            "critical": esc["key"] in CRITICAL,
            "extracted": str(entry.get("value", ""))[:28],
            "conf": round(esc["confidence"], 2),
            "threshold": entry.get("threshold"),
            "grounded": entry.get("grounded"),
            "reasons": ",".join(esc["reasons"]),
        })
queue = pd.DataFrame(queue_rows)
print(f"{len(queue)} fields flagged across {queue['doc'].nunique() if len(queue) else 0} documents")
print(f"estimated review time: {len(results) and sum(1 for r in results if not r['straight_through'])} "
      f"documents x {CFG.review_seconds_flagged}s = "
      f"{sum(1 for r in results if not r['straight_through']) * CFG.review_seconds_flagged / 60:.1f} min")
print(f"fully manual equivalent: {len(results) * CFG.review_seconds_manual / 60:.1f} min")
queue.head(30) if len(queue) else "nothing flagged"


In [ ]:
if len(queue):
    reasons = Counter(r for row in queue["reasons"] for r in row.split(","))
    fig, ax = plt.subplots(figsize=(9, 3.2))
    names = [n for n, _ in reasons.most_common()]
    counts = [c for _, c in reasons.most_common()]
    ax.barh(range(len(names)), counts, color=viz.SEQ_HUE, height=0.6)
    for i, c in enumerate(counts):
        ax.text(c + 0.3, i, str(c), va="center", color=viz.INK_2, fontsize=9)
    ax.set_yticks(range(len(names)), names)
    ax.invert_yaxis()
    ax.set_xlabel("fields flagged")
    ax.set_title("Why the Validator escalated")
    ax.grid(axis="y", visible=False)
    plt.tight_layout(); plt.show()
else:
    print("no escalations to chart")

## The PFTFI loop in action

Corrections are taken from the labelled ground truth, standing in for the human
reviewer. That is legitimate for demonstrating the mechanism — the labels *are*
dual-annotated human output — but note what it cannot show: whether accuracy
improves on **unseen** documents. Feeding a document its own answer and then
re-scoring that same document proves nothing.

So the loop is run properly: corrections are captured on one half of the
documents, and the effect is measured on the **other** half, which the store has
never seen.

In [ ]:
expected_all: dict[str, dict[str, Any]] = {}
for case in config.case_dirs():
    path = config.expected_fields_path(case)
    if path.is_file():
        for name, payload in json.loads(path.read_text(encoding="utf-8")).items():
            expected_all[f"{case.name}/{name}"] = payload

scored = [r for r in results if expected_all.get(r["doc_id"], {}).get("fields")]
teach, hold = scored[::2], scored[1::2]
print(f"{len(teach)} documents to teach from, {len(hold)} held out\n")

def corrections_from(record: dict[str, Any]) -> list[Correction]:
    truth = expected_all.get(record["doc_id"], {}).get("fields", {})
    kind = record["routing"]["kind"]
    out = []
    for key, want in truth.items():
        got = record["fields"].get(key, {}).get("value")
        if got is None or normalize.platform_norm(str(got)) != normalize.platform_norm(str(want)):
            # A value the OCR never produced is a layout/recognition failure;
            # a wrong value from present text is a semantic one.
            cause = "layout" if not normalize.value_in_text(str(want), record["_text"]) else "semantic"
            out.append(Correction(record["doc_id"], kind, key, got, str(want), cause))
    return out

store = PFTFI()
for record in teach:
    store.capture(corrections_from(record))

print(f"PFTFI version {store.version}, {len(store.corrections)} corrections captured")
print(f"  semantic: {sum(1 for c in store.corrections if c.cause == 'semantic')}")
print(f"  layout  : {sum(1 for c in store.corrections if c.cause == 'layout')}")
print(f"  reusable patterns learned: {sum(len(v) for v in store.learned_patterns.values())}")
print(f"  parser hints: {len(store.parser_hints)}")
for hint in store.parser_hints[:4]:
    print("   -", hint)

def score_fields(record: dict[str, Any], fields: dict[str, dict[str, Any]]) -> tuple[int, int]:
    truth = expected_all.get(record["doc_id"], {}).get("fields", {})
    correct = sum(
        1 for k, want in truth.items()
        if fields.get(k, {}).get("value") is not None
        and normalize.platform_norm(str(fields[k]["value"])) == normalize.platform_norm(str(want))
    )
    return correct, len(truth)

before = after = total = 0
for record in hold:
    keys = field_keys_for(record["routing"]["kind"])
    b, n = score_fields(record, record["fields"])
    inherited = store.apply_inheritance(record["_text"], keys, record["fields"])
    a, _ = score_fields(record, inherited)
    before, after, total = before + b, after + a, total + n

print(f"\nheld-out documents: {before}/{total} correct before inheritance, "
      f"{after}/{total} after")
if total:
    print(f"delta: {(after - before) / total:+.1%}")

## Metrics

The paper's four measures, defined as it defines them:

- **Automation rate** — documents reaching output without escalation ÷ total
- **STP** — documents with zero escalations from *any* agent (the strictest)
- **Document-level accuracy** — documents where *all* required fields are correct
- **Human intervention rate** — fraction routed to the review queue

Field-level precision, recall and F1 are added because document-level accuracy is
brutal on a 20-field form: one wrong field fails the whole document.

In [ ]:
def compute_metrics(records: list[dict[str, Any]], label: str = "") -> dict[str, Any]:
    n = len(records)
    if not n:
        return {}
    stp = sum(1 for r in records if r["straight_through"])
    escalated = n - stp

    tp = fp = fn = 0
    docs_perfect = docs_scored = 0
    for r in records:
        truth = expected_all.get(r["doc_id"], {}).get("fields", {})
        if not truth:
            continue
        docs_scored += 1
        correct, total = score_fields(r, r["fields"])
        predicted = sum(1 for k in truth if r["fields"].get(k, {}).get("value") is not None)
        tp += correct
        fp += predicted - correct
        fn += total - predicted
        docs_perfect += int(correct == total and total > 0)

    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    return {
        "configuration": label,
        "documents": n,
        "automation_rate": stp / n,
        "stp": stp / n,
        "human_intervention_rate": escalated / n,
        "doc_accuracy": docs_perfect / docs_scored if docs_scored else None,
        "docs_scored": docs_scored,
        "field_precision": precision,
        "field_recall": recall,
        "field_f1": f1,
        "seconds_per_doc": float(np.mean([r["seconds"] for r in records])),
        "pages": sum(r["pages"] for r in records),
    }

current = compute_metrics(results, "Full MADP (offline backend)")
for key, value in current.items():
    if isinstance(value, float):
        print(f"  {key:26s} {value:.3f}" if value <= 1 else f"  {key:26s} {value:.1f}")
    else:
        print(f"  {key:26s} {value}")

## Ablation — reproducing the paper's Table 2 shape

The paper adds one agent at a time and reports accuracy:

| Configuration | Paper accuracy |
|---|---|
| Baseline (direct LLM on PDF) | 60.0% |
| + Classificator | 65.0% |
| + Splitter | 72.5% |
| + Parser | 90.0% |
| + Validator + PFTFI | 92.5% |
| Full MADP + HITL | 98.5% |

The same ladder is run here. **The absolute numbers will not match and should not
be expected to** — different domain, no LLM backend, Sinhala pages the OCR cannot
read. What is worth reading is the *shape*: which components carry their weight
on this corpus.

In [ ]:
LADDER = [
    ("Baseline (no agents)",      dict(use_classificator=False, use_splitter=False, use_parser=False, use_validator=False, use_hitl=False)),
    ("+ Classificator",           dict(use_classificator=True,  use_splitter=False, use_parser=False, use_validator=False, use_hitl=False)),
    ("+ Splitter",                dict(use_classificator=True,  use_splitter=True,  use_parser=False, use_validator=False, use_hitl=False)),
    ("+ Parser",                  dict(use_classificator=True,  use_splitter=True,  use_parser=True,  use_validator=False, use_hitl=False)),
    ("+ Validator + PFTFI",       dict(use_classificator=True,  use_splitter=True,  use_parser=True,  use_validator=True,  use_hitl=False)),
    ("Full MADP + HITL",          dict(use_classificator=True,  use_splitter=True,  use_parser=True,  use_validator=True,  use_hitl=True)),
]

ablation_rows = []
for label, flags in LADDER:
    cfg = MADPConfig(**flags)
    store_i = PFTFI()
    records = []
    for doc in documents:
        rec = process_document(doc, cfg, store_i if cfg.use_validator else None)
        records.append(rec)
    if cfg.use_hitl:
        # HITL: the reviewer corrects everything the validator flagged. This is
        # the paper's "Full MADP + HITL" row - accuracy after human correction,
        # which is why it is an upper bound rather than an automation result.
        for rec in records:
            truth = expected_all.get(rec["doc_id"], {}).get("fields", {})
            flagged = {e["key"] for e in rec["validation"]["escalations"]}
            for key in flagged & set(truth):
                rec["fields"][key] = {"value": truth[key], "confidence": 1.0,
                                      "backend": "human", "grounded": True}
    m = compute_metrics(records, label)
    ablation_rows.append(m)
    print(f"{label:26s} doc_acc={str(round(m['doc_accuracy'],3) if m['doc_accuracy'] is not None else 'n/a'):>6s} "
          f"F1={m['field_f1']:.3f}  STP={m['stp']:.2f}  {m['seconds_per_doc']:.1f}s/doc")

ablation = pd.DataFrame(ablation_rows)
ablation[["configuration", "doc_accuracy", "field_precision", "field_recall",
          "field_f1", "stp", "human_intervention_rate", "seconds_per_doc"]]

In [ ]:
PAPER = [0.600, 0.650, 0.725, 0.900, 0.925, 0.985]
labels = [r["configuration"] for r in ablation_rows]
ours = [r["field_f1"] for r in ablation_rows]
x = np.arange(len(labels))

fig, axes = plt.subplots(1, 2, figsize=(14, 4.6))

# Two measures on different footings, so two panels rather than one dual axis.
ax = axes[0]
steps = [viz.SEQ_STEPS[min(i + 1, len(viz.SEQ_STEPS) - 1)] for i in range(len(labels))]
ax.bar(x, ours, 0.62, color=steps)
for i, v in enumerate(ours):
    ax.text(i, v + 0.012, f"{v:.2f}", ha="center", color=viz.INK_2, fontsize=9)
ax.set_xticks(x, labels, rotation=30, ha="right")
ax.set_ylabel("field F1")
ax.set_ylim(0, max(0.4, max(ours) * 1.35))
ax.set_title("This corpus — field F1 as agents are added")

ax = axes[1]
ax.plot(x, PAPER, marker="o", color=viz.SERIES[1], label="paper (invoices, with LLM)")
ax.plot(x, ours, marker="o", color=viz.SERIES[0], label="here (scans, no LLM)")
ax.set_xticks(x, labels, rotation=30, ha="right")
ax.set_ylabel("accuracy / F1")
ax.set_ylim(0, 1.05)
ax.set_title("Shape comparison — not a like-for-like benchmark")
ax.legend(loc="upper left")

plt.tight_layout(); plt.show()

gains = [(labels[i], ours[i] - ours[i - 1]) for i in range(1, len(ours))]
print("incremental gain per component (field F1):")
for name, delta in gains:
    print(f"  {name:26s} {delta:+.3f}")
best = max(gains, key=lambda g: g[1]) if gains else None
if best:
    print(f"\nlargest contributor here: {best[0]} ({best[1]:+.3f})")
    print("paper's largest contributor: + Parser Agent (+0.175)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.2))

ax = axes[0]
stp_vals = [r["stp"] for r in ablation_rows]
hir_vals = [r["human_intervention_rate"] for r in ablation_rows]
ax.bar(x - 0.2, stp_vals, 0.36, color=viz.STATUS["correct"], label="straight-through")
ax.bar(x + 0.2, hir_vals, 0.36, color=viz.STATUS["missing"], label="needs a human")
ax.set_xticks(x, labels, rotation=30, ha="right")
ax.set_ylabel("share of documents")
ax.set_title("Automation vs human intervention")
ax.legend()

ax = axes[1]
stages = ["render", "parse", "split", "classify", "extract", "validate"]
totals = {s: sum(r["timings"].get(s, 0) for r in results) for s in stages}
left = 0.0
for i, stage in enumerate(stages):
    v = totals[stage]
    if v <= 0:
        continue
    ax.barh(0, v, left=left, height=0.5, color=viz.SEQ_STEPS[min(i + 1, 6)],
            edgecolor=viz.SURFACE, linewidth=2, label=stage)
    left += v
ax.set_yticks([])
ax.set_xlabel("seconds across the run")
ax.set_title("Where the time goes")
ax.legend(ncol=3, loc="upper center", bbox_to_anchor=(0.5, -0.25))
ax.grid(axis="y", visible=False)

plt.tight_layout(); plt.show()
print({k: round(v, 1) for k, v in totals.items()})

## Reading the result honestly

Run the cells above and the numbers will be low. That is the finding, and it is a
useful one.

**What this run actually measures.** MADP's architecture is fully implemented and
runs end to end on real matter bundles: five agents, consensus voting,
deterministic atomic checks with confidence elevation, a real escalation queue,
and a PFTFI loop evaluated on held-out documents. What it is *not* measuring is
MADP as the authors built it, because two of the paper's load-bearing assumptions
do not hold here:

1. **The paper's documents have a text layer; these do not.** Docling's layout
   analysis — the source of its largest gain — has nothing to analyse on a photo
   of a deed. The Parser Agent here does the equivalent work over OCR output.
2. **The paper's extraction is LLM-driven; there is no reachable LLM.** The rule
   backend is a floor, not a substitute. Its recall on free-form legal prose is
   poor by construction.

On top of that, the available OCR engine reads Latin script only. Sinhala pages —
a large part of this corpus — come back as noise, so any field on a Sinhala page
is unreachable regardless of how good the downstream agents are.

**What is genuinely transferable.** Three ideas earn their place independent of
the missing pieces:

- **Atomic consistency checks that elevate confidence.** Cheap, deterministic,
  and they catch the failure that matters most here — an instrument transferring
  more land than the parcel contains is caught by arithmetic, not by a model.
- **Splitting before classifying.** Production classifies a whole document from
  page 1, so a 16-page bundle is read under one schema. The splitter is the fix,
  and it needs no LLM.
- **Escalation as a first-class output.** "I am not confident" routed to a lawyer
  beats a confident wrong parcel number, which is the same conclusion the
  benchmark's invented-value metric reaches from the other direction.

**What to do next, in order.** Fix the Gemini credential and re-run — that alone
changes the extraction floor. Add a Sinhala-capable OCR path, since no amount of
agent architecture recovers text that was never recognised. Only then is the
ablation above worth using to decide anything.

Nothing here is written to a tracked file: extracted values and page text stay in
memory. **Clear outputs before committing** — the tables above contain real
client values.